In [7]:
# ---------------------------------------------
# Hospital Emergency Room Optimizer (FINAL)
# ---------------------------------------------

import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder

print("Program Started...\n")

# ---------------------------------------------
# Step 1: Load Dataset from CSV
# ---------------------------------------------

data = pd.read_csv("hospital_data.csv")

print("Dataset Loaded:\n")
print(data.head())

# ---------------------------------------------
# Step 2: Encode Labels
# ---------------------------------------------

le = LabelEncoder()
data["SeverityEncoded"] = le.fit_transform(data["Severity"])

X = data[["Fever","HeartRate","BP","Oxygen","PainLevel"]]
y = data["SeverityEncoded"]

# ---------------------------------------------
# Step 3: Train Model
# ---------------------------------------------

model = LogisticRegression(max_iter=1000)
model.fit(X, y)

print("\nModel trained successfully!\n")

# ---------------------------------------------
# Step 4: Test Patient (Fixed Input)
# ---------------------------------------------

print("Testing with sample patient...\n")

test_patient = pd.DataFrame([[0, 80, 120, 98, 2]],
                           columns=["Fever","HeartRate","BP","Oxygen","PainLevel"])

prediction = model.predict(test_patient)
severity_result = le.inverse_transform(prediction)

print("Predicted Severity:", severity_result[0])

# ---------------------------------------------
# Step 5: Greedy Doctor Assignment (Search)
# ---------------------------------------------

doctors = {
    "High": "Senior Doctor - Room 101",
    "Medium": "General Doctor - Room 102",
    "Low": "Junior Doctor - Room 103"
}

assigned = doctors[severity_result[0]]

print("Assigned:", assigned)

print("\nProgram Finished ✅")

Program Started...

Dataset Loaded:

   Fever  HeartRate   BP  Oxygen  PainLevel Severity
0      1        130   85      82          9     High
1      0         80  120      98          2      Low
2      1        110   90      88          7   Medium
3      1        140   80      78         10     High
4      0         75  130      99          1      Low

Model trained successfully!

Testing with sample patient...

Predicted Severity: Low
Assigned: Junior Doctor - Room 103

Program Finished ✅


In [9]:
"""
Hospital Emergency Room Optimizer
==================================
Uses AI/ML & Search Techniques:
  - Priority Queue (Heap) for triage
  - Greedy Scheduling for doctor assignment
  - A* Search for bed routing
  - Simulated Annealing for shift optimization
  - Decision Tree logic for severity classification
"""

import heapq
import random
import math
import time
from datetime import datetime, timedelta
from collections import defaultdict


# ─────────────────────────────────────────────
# 1. PATIENT & TRIAGE SYSTEM
# ─────────────────────────────────────────────

SEVERITY_LEVELS = {
    1: ("CRITICAL",   "🔴", 0),    # Immediate
    2: ("EMERGENT",   "🟠", 15),   # Within 15 min
    3: ("URGENT",     "🟡", 30),   # Within 30 min
    4: ("LESS URGENT","🟢", 60),   # Within 60 min
    5: ("NON-URGENT", "⚪", 120),  # Within 2 hrs
}

SYMPTOMS_DB = {
    "chest pain":        1,
    "cardiac arrest":    1,
    "severe bleeding":   1,
    "stroke symptoms":   1,
    "unconscious":       1,
    "difficulty breathing": 2,
    "high fever":        2,
    "severe allergic":   2,
    "broken bone":       3,
    "abdominal pain":    3,
    "deep laceration":   3,
    "mild fever":        4,
    "sprain":            4,
    "ear pain":          5,
    "common cold":       5,
    "minor rash":        5,
}

patient_counter = 0

class Patient:
    def __init__(self, name, age, symptoms, arrival_time=None):
        global patient_counter
        patient_counter += 1
        self.id = patient_counter
        self.name = name
        self.age = age
        self.symptoms = symptoms
        self.arrival_time = arrival_time or datetime.now()
        self.severity = self._classify_severity()
        self.assigned_doctor = None
        self.bed_number = None
        self.wait_start = self.arrival_time

    def _classify_severity(self):
        """Decision Tree logic: classify based on symptoms + age risk"""
        max_sev = 5
        for symptom in self.symptoms:
            s = symptom.lower()
            for key, level in SYMPTOMS_DB.items():
                if key in s:
                    max_sev = min(max_sev, level)

        # Age-based risk adjustment (elderly/children get bumped up)
        if self.age > 70 or self.age < 5:
            max_sev = max(1, max_sev - 1)

        return max_sev

    def wait_time_minutes(self):
        delta = datetime.now() - self.wait_start
        return int(delta.total_seconds() / 60)

    def __lt__(self, other):
        # Lower severity number = higher priority
        if self.severity != other.severity:
            return self.severity < other.severity
        return self.arrival_time < other.arrival_time  # FIFO within same level

    def __str__(self):
        sev_name, icon, _ = SEVERITY_LEVELS[self.severity]
        return (f"[P{self.id:03d}] {self.name} (age {self.age}) | "
                f"{icon} {sev_name} | Symptoms: {', '.join(self.symptoms)}")


# ─────────────────────────────────────────────
# 2. PRIORITY QUEUE TRIAGE SYSTEM
# ─────────────────────────────────────────────

class TriageQueue:
    def __init__(self):
        self._heap = []
        self._count = 0  # tie-breaker

    def add_patient(self, patient):
        # heap entry: (severity, counter, patient)
        heapq.heappush(self._heap, (patient.severity, self._count, patient))
        self._count += 1
        print(f"\n  ✅ Admitted: {patient}")

    def next_patient(self):
        if self._heap:
            _, _, patient = heapq.heappop(self._heap)
            return patient
        return None

    def peek(self):
        if self._heap:
            return self._heap[0][2]
        return None

    def size(self):
        return len(self._heap)

    def all_patients(self):
        return sorted([entry[2] for entry in self._heap])

    def is_empty(self):
        return len(self._heap) == 0


# ─────────────────────────────────────────────
# 3. DOCTOR & BED MANAGEMENT
# ─────────────────────────────────────────────

class Doctor:
    def __init__(self, name, specialization, experience_years):
        self.name = name
        self.specialization = specialization
        self.experience = experience_years
        self.current_patient = None
        self.patients_treated = 0
        self.busy_until = datetime.now()

    def is_available(self):
        return datetime.now() >= self.busy_until

    def assign_patient(self, patient, duration_minutes=20):
        self.current_patient = patient
        self.busy_until = datetime.now() + timedelta(minutes=duration_minutes)
        self.patients_treated += 1
        patient.assigned_doctor = self.name

    def __str__(self):
        status = "🟢 Available" if self.is_available() else "🔴 Busy"
        return f"Dr. {self.name} [{self.specialization}] | {status} | Treated: {self.patients_treated}"


class BedManager:
    def __init__(self, total_beds):
        self.total_beds = total_beds
        # beds dict: bed_num -> patient or None
        self.beds = {i: None for i in range(1, total_beds + 1)}

    def assign_bed(self, patient):
        """A*-inspired: prefer beds closest to nurse station (lower number = closer)"""
        for bed_num in sorted(self.beds.keys()):
            if self.beds[bed_num] is None:
                self.beds[bed_num] = patient
                patient.bed_number = bed_num
                return bed_num
        return None  # No beds available

    def release_bed(self, bed_num):
        if bed_num in self.beds:
            self.beds[bed_num] = None

    def available_beds(self):
        return sum(1 for v in self.beds.values() if v is None)

    def occupancy_rate(self):
        occupied = self.total_beds - self.available_beds()
        return (occupied / self.total_beds) * 100

    def display(self):
        print("\n  🛏️  BED STATUS:")
        for bed, patient in self.beds.items():
            status = f"  Patient: {patient.name}" if patient else "  [Empty]"
            icon = "🔴" if patient else "🟢"
            print(f"    Bed {bed:2d}: {icon}{status}")


# ─────────────────────────────────────────────
# 4. GREEDY DOCTOR ASSIGNMENT
# ─────────────────────────────────────────────

def assign_best_doctor(patient, doctors):
    """
    Greedy Search: Pick best available doctor.
    Score = experience * 10 - (1 if specialization mismatch)
    """
    best_doctor = None
    best_score = -1

    for doc in doctors:
        if not doc.is_available():
            continue

        score = doc.experience * 10

        # Bonus for matching specialization to severity
        if patient.severity <= 2 and "Emergency" in doc.specialization:
            score += 50
        elif patient.severity == 3 and "Surgery" in doc.specialization:
            score += 30

        if score > best_score:
            best_score = score
            best_doctor = doc

    return best_doctor


# ─────────────────────────────────────────────
# 5. SIMULATED ANNEALING - SHIFT OPTIMIZER
# ─────────────────────────────────────────────

def shift_optimizer(doctors, num_slots=3, iterations=500):
    """
    Simulated Annealing to optimize doctor shift assignments.
    Minimizes coverage gaps and maximizes specialization coverage.
    """
    slots = ["Morning", "Afternoon", "Night"]

    # Initial random assignment
    assignment = {doc.name: random.choice(slots) for doc in doctors}

    def cost(assign):
        """Lower is better: penalize unbalanced shifts"""
        slot_counts = defaultdict(int)
        for slot in assign.values():
            slot_counts[slot] += 1
        variance = sum((count - len(doctors)/num_slots)**2 for count in slot_counts.values())
        # Penalize if no emergency doc on night shift
        night_docs = [d for d in doctors if assign[d.name] == "Night" and "Emergency" in d.specialization]
        penalty = 100 if not night_docs else 0
        return variance + penalty

    current_cost = cost(assignment)
    T = 100.0  # Temperature

    for i in range(iterations):
        # Pick random doctor and random new slot
        doc = random.choice(doctors)
        new_slot = random.choice(slots)
        old_slot = assignment[doc.name]
        assignment[doc.name] = new_slot

        new_cost = cost(assignment)
        delta = new_cost - current_cost

        # Accept if better, or probabilistically if worse
        if delta < 0 or random.random() < math.exp(-delta / T):
            current_cost = new_cost
        else:
            assignment[doc.name] = old_slot  # Revert

        T *= 0.995  # Cool down

    return assignment


# ─────────────────────────────────────────────
# 6. ER STATISTICS & REPORTING
# ─────────────────────────────────────────────

class ERStats:
    def __init__(self):
        self.treated = []
        self.wait_times = []

    def record(self, patient, wait_minutes):
        self.treated.append(patient)
        self.wait_times.append(wait_minutes)

    def report(self):
        print("\n" + "="*55)
        print("         📊  ER SESSION STATISTICS")
        print("="*55)
        if not self.treated:
            print("  No patients treated yet.")
            return

        total = len(self.treated)
        avg_wait = sum(self.wait_times) / total if total else 0
        max_wait = max(self.wait_times) if self.wait_times else 0

        severity_counts = defaultdict(int)
        for p in self.treated:
            severity_counts[p.severity] += 1

        print(f"  Total patients treated : {total}")
        print(f"  Average wait time      : {avg_wait:.1f} min")
        print(f"  Max wait time          : {max_wait} min")
        print("\n  Severity Breakdown:")
        for sev in sorted(severity_counts):
            name, icon, _ = SEVERITY_LEVELS[sev]
            print(f"    {icon} Level {sev} ({name:12s}): {severity_counts[sev]} patients")
        print("="*55)


# ─────────────────────────────────────────────
# 7. MAIN ER SIMULATION
# ─────────────────────────────────────────────

class EmergencyRoom:
    def __init__(self, num_beds=10):
        self.triage = TriageQueue()
        self.bed_manager = BedManager(num_beds)
        self.stats = ERStats()

        # Initialize doctors
        self.doctors = [
            Doctor("Smith",   "Emergency Medicine", 15),
            Doctor("Patel",   "Emergency Medicine", 8),
            Doctor("Chen",    "Surgery",            12),
            Doctor("Nguyen",  "Cardiology",         20),
            Doctor("Garcia",  "Pediatrics",         10),
        ]

    def admit_patient(self, name, age, symptoms):
        patient = Patient(name, age, symptoms)
        self.triage.add_patient(patient)
        return patient

    def process_next_patient(self):
        if self.triage.is_empty():
            print("\n  ℹ️  No patients in queue.")
            return None

        patient = self.triage.next_patient()
        print(f"\n  🏥 Processing: {patient}")

        # Assign bed
        bed = self.bed_manager.assign_bed(patient)
        if bed:
            print(f"  🛏️  Assigned to Bed {bed}")
        else:
            print("  ⚠️  No beds available! Patient in waiting area.")

        # Assign doctor
        doctor = assign_best_doctor(patient, self.doctors)
        if doctor:
            wait = patient.wait_time_minutes()
            doctor.assign_patient(patient)
            self.stats.record(patient, wait)
            print(f"  👨‍⚕️  Assigned to Dr. {doctor.name} ({doctor.specialization})")
            print(f"  ⏱️  Wait time: {wait} min")
        else:
            print("  ⚠️  All doctors busy. Patient waiting for doctor.")

        return patient

    def show_queue(self):
        patients = self.triage.all_patients()
        print(f"\n  📋 TRIAGE QUEUE ({len(patients)} patients):")
        if not patients:
            print("    [Queue empty]")
        for i, p in enumerate(patients, 1):
            sev_name, icon, max_wait = SEVERITY_LEVELS[p.severity]
            print(f"    {i}. {icon} {p.name:15s} | {sev_name:12s} | Max wait: {max_wait} min")

    def show_doctors(self):
        print("\n  👨‍⚕️  DOCTOR STATUS:")
        for doc in self.doctors:
            print(f"    {doc}")

    def show_shift_plan(self):
        print("\n  📅 OPTIMIZED SHIFT PLAN (Simulated Annealing):")
        assignment = shift_optimizer(self.doctors)
        shifts = defaultdict(list)
        for name, slot in assignment.items():
            shifts[slot].append(name)
        for slot in ["Morning", "Afternoon", "Night"]:
            docs = ", ".join(f"Dr. {n}" for n in shifts[slot])
            print(f"    {slot:12s}: {docs}")

    def run_demo(self):
        """Full demo run"""
        print("\n" + "="*55)
        print("   🏥  HOSPITAL EMERGENCY ROOM OPTIMIZER")
        print("        AI/ML + Search Techniques")
        print("="*55)

        # --- Admit patients ---
        print("\n📥 ADMITTING PATIENTS...\n" + "-"*40)
        patients_data = [
            ("Alice Kumar",  34, ["chest pain", "difficulty breathing"]),
            ("Bob Singh",    72, ["mild fever", "sprain"]),
            ("Carol Mehta",  5,  ["high fever", "ear pain"]),
            ("David Rao",    45, ["severe bleeding"]),
            ("Eva Sharma",   28, ["abdominal pain"]),
            ("Frank Nair",   60, ["common cold"]),
            ("Grace Thomas", 80, ["stroke symptoms"]),
            ("Harish Pillai",22, ["broken bone"]),
        ]

        for name, age, symptoms in patients_data:
            self.admit_patient(name, age, symptoms)
            time.sleep(0.05)

        # --- Show queue ---
        self.show_queue()

        # --- Process all patients ---
        print("\n\n🔄 PROCESSING PATIENTS (Priority Order)...\n" + "-"*40)
        while not self.triage.is_empty():
            self.process_next_patient()
            time.sleep(0.1)

        # --- Bed status ---
        self.bed_manager.display()

        # --- Doctor status ---
        self.show_doctors()

        # --- Shift optimization ---
        self.show_shift_plan()

        # --- Stats report ---
        self.stats.report()


# ─────────────────────────────────────────────
# 8. INTERACTIVE MENU
# ─────────────────────────────────────────────

def interactive_mode(er):
    while True:
        print("\n" + "─"*45)
        print("  🏥  ER OPTIMIZER — MENU")
        print("─"*45)
        print("  1. Admit new patient")
        print("  2. Process next patient")
        print("  3. Show triage queue")
        print("  4. Show doctor status")
        print("  5. Show bed status")
        print("  6. Show shift plan")
        print("  7. Show statistics")
        print("  0. Exit")
        print("─"*45)
        choice = input("  Enter choice: ").strip()

        if choice == "1":
            name = input("  Patient name: ").strip()
            try:
                age = int(input("  Age: "))
            except ValueError:
                print("  Invalid age."); continue
            syms = input("  Symptoms (comma-separated): ").strip().split(",")
            syms = [s.strip() for s in syms if s.strip()]
            er.admit_patient(name, age, syms)

        elif choice == "2":
            er.process_next_patient()

        elif choice == "3":
            er.show_queue()

        elif choice == "4":
            er.show_doctors()

        elif choice == "5":
            er.bed_manager.display()

        elif choice == "6":
            er.show_shift_plan()

        elif choice == "7":
            er.stats.report()

        elif choice == "0":
            print("\n  👋 Exiting ER Optimizer. Goodbye!\n")
            break
        else:
            print("  ❌ Invalid choice.")


# ─────────────────────────────────────────────
# ENTRY POINT
# ─────────────────────────────────────────────

if __name__ == "__main__":
    er = EmergencyRoom(num_beds=10)

    print("\nChoose mode:")
    print("  1. Run full demo (recommended first time)")
    print("  2. Interactive mode")
    mode = input("Enter 1 or 2: ").strip()

    if mode == "1":
        er.run_demo()
    elif mode == "2":
        interactive_mode(er)
    else:
        print("Invalid choice. Running demo by default.")
        er.run_demo()


Choose mode:
  1. Run full demo (recommended first time)
  2. Interactive mode


Enter 1 or 2:  1



   🏥  HOSPITAL EMERGENCY ROOM OPTIMIZER
        AI/ML + Search Techniques

📥 ADMITTING PATIENTS...
----------------------------------------

  ✅ Admitted: [P001] Alice Kumar (age 34) | 🔴 CRITICAL | Symptoms: chest pain, difficulty breathing

  ✅ Admitted: [P002] Bob Singh (age 72) | 🟡 URGENT | Symptoms: mild fever, sprain

  ✅ Admitted: [P003] Carol Mehta (age 5) | 🟠 EMERGENT | Symptoms: high fever, ear pain

  ✅ Admitted: [P004] David Rao (age 45) | 🔴 CRITICAL | Symptoms: severe bleeding

  ✅ Admitted: [P005] Eva Sharma (age 28) | 🟡 URGENT | Symptoms: abdominal pain

  ✅ Admitted: [P006] Frank Nair (age 60) | ⚪ NON-URGENT | Symptoms: common cold

  ✅ Admitted: [P007] Grace Thomas (age 80) | 🔴 CRITICAL | Symptoms: stroke symptoms

  ✅ Admitted: [P008] Harish Pillai (age 22) | 🟡 URGENT | Symptoms: broken bone

  📋 TRIAGE QUEUE (8 patients):
    1. 🔴 Alice Kumar     | CRITICAL     | Max wait: 0 min
    2. 🔴 David Rao       | CRITICAL     | Max wait: 0 min
    3. 🔴 Grace Thomas    | CRIT

In [3]:
"""
Hospital Emergency Room Optimizer
==================================
Uses AI/ML & Search Techniques:
  - Priority Queue (Heap) for triage
  - Greedy Scheduling for doctor assignment
  - A* Search for bed routing
  - Simulated Annealing for shift optimization
  - Decision Tree logic for severity classification
"""

import heapq
import random
import math
import time
from datetime import datetime, timedelta
from collections import defaultdict


# ─────────────────────────────────────────────
# 1. PATIENT & TRIAGE SYSTEM
# ─────────────────────────────────────────────

SEVERITY_LEVELS = {
    1: ("CRITICAL",   "🔴", 0),    # Immediate
    2: ("EMERGENT",   "🟠", 15),   # Within 15 min
    3: ("URGENT",     "🟡", 30),   # Within 30 min
    4: ("LESS URGENT","🟢", 60),   # Within 60 min
    5: ("NON-URGENT", "⚪", 120),  # Within 2 hrs
}

SYMPTOMS_DB = {
    "chest pain":        1,
    "cardiac arrest":    1,
    "severe bleeding":   1,
    "stroke symptoms":   1,
    "unconscious":       1,
    "difficulty breathing": 2,
    "high fever":        2,
    "severe allergic":   2,
    "broken bone":       3,
    "abdominal pain":    3,
    "deep laceration":   3,
    "mild fever":        4,
    "sprain":            4,
    "ear pain":          5,
    "common cold":       5,
    "minor rash":        5,
}

patient_counter = 0

class Patient:
    def __init__(self, name, age, symptoms, arrival_time=None):
        global patient_counter
        patient_counter += 1
        self.id = patient_counter
        self.name = name
        self.age = age
        self.symptoms = symptoms
        self.arrival_time = arrival_time or datetime.now()
        self.severity = self._classify_severity()
        self.assigned_doctor = None
        self.bed_number = None
        self.wait_start = self.arrival_time

    def _classify_severity(self):
        """Decision Tree logic: classify based on symptoms + age risk"""
        max_sev = 5
        for symptom in self.symptoms:
            s = symptom.lower()
            for key, level in SYMPTOMS_DB.items():
                if key in s:
                    max_sev = min(max_sev, level)

        # Age-based risk adjustment (elderly/children get bumped up)
        if self.age > 70 or self.age < 5:
            max_sev = max(1, max_sev - 1)

        return max_sev

    def wait_time_minutes(self):
        delta = datetime.now() - self.wait_start
        return int(delta.total_seconds() / 60)

    def __lt__(self, other):
        # Lower severity number = higher priority
        if self.severity != other.severity:
            return self.severity < other.severity
        return self.arrival_time < other.arrival_time  # FIFO within same level

    def __str__(self):
        sev_name, icon, _ = SEVERITY_LEVELS[self.severity]
        return (f"[P{self.id:03d}] {self.name} (age {self.age}) | "
                f"{icon} {sev_name} | Symptoms: {', '.join(self.symptoms)}")


# ─────────────────────────────────────────────
# 2. PRIORITY QUEUE TRIAGE SYSTEM
# ─────────────────────────────────────────────

class TriageQueue:
    def __init__(self):
        self._heap = []
        self._count = 0  # tie-breaker

    def add_patient(self, patient):
        # heap entry: (severity, counter, patient)
        heapq.heappush(self._heap, (patient.severity, self._count, patient))
        self._count += 1
        print(f"\n  ✅ Admitted: {patient}")

    def next_patient(self):
        if self._heap:
            _, _, patient = heapq.heappop(self._heap)
            return patient
        return None

    def peek(self):
        if self._heap:
            return self._heap[0][2]
        return None

    def size(self):
        return len(self._heap)

    def all_patients(self):
        return sorted([entry[2] for entry in self._heap])

    def is_empty(self):
        return len(self._heap) == 0


# ─────────────────────────────────────────────
# 3. DOCTOR & BED MANAGEMENT
# ─────────────────────────────────────────────

class Doctor:
    def __init__(self, name, specialization, experience_years):
        self.name = name
        self.specialization = specialization
        self.experience = experience_years
        self.current_patient = None
        self.patients_treated = 0
        self.busy_until = datetime.now()

    def is_available(self):
        return datetime.now() >= self.busy_until

    def assign_patient(self, patient, duration_minutes=20):
        self.current_patient = patient
        self.busy_until = datetime.now() + timedelta(minutes=duration_minutes)
        self.patients_treated += 1
        patient.assigned_doctor = self.name

    def __str__(self):
        status = "🟢 Available" if self.is_available() else "🔴 Busy"
        return f"Dr. {self.name} [{self.specialization}] | {status} | Treated: {self.patients_treated}"


class BedManager:
    def __init__(self, total_beds):
        self.total_beds = total_beds
        # beds dict: bed_num -> patient or None
        self.beds = {i: None for i in range(1, total_beds + 1)}

    def assign_bed(self, patient):
        """A*-inspired: prefer beds closest to nurse station (lower number = closer)"""
        for bed_num in sorted(self.beds.keys()):
            if self.beds[bed_num] is None:
                self.beds[bed_num] = patient
                patient.bed_number = bed_num
                return bed_num
        return None  # No beds available

    def release_bed(self, bed_num):
        if bed_num in self.beds:
            self.beds[bed_num] = None

    def available_beds(self):
        return sum(1 for v in self.beds.values() if v is None)

    def occupancy_rate(self):
        occupied = self.total_beds - self.available_beds()
        return (occupied / self.total_beds) * 100

    def display(self):
        print("\n  🛏️  BED STATUS:")
        for bed, patient in self.beds.items():
            status = f"  Patient: {patient.name}" if patient else "  [Empty]"
            icon = "🔴" if patient else "🟢"
            print(f"    Bed {bed:2d}: {icon}{status}")


# ─────────────────────────────────────────────
# 4. GREEDY DOCTOR ASSIGNMENT
# ─────────────────────────────────────────────

def assign_best_doctor(patient, doctors):
    """
    Greedy Search: Pick best available doctor.
    Score = experience * 10 - (1 if specialization mismatch)
    """
    best_doctor = None
    best_score = -1

    for doc in doctors:
        if not doc.is_available():
            continue

        score = doc.experience * 10

        # Bonus for matching specialization to severity
        if patient.severity <= 2 and "Emergency" in doc.specialization:
            score += 50
        elif patient.severity == 3 and "Surgery" in doc.specialization:
            score += 30

        if score > best_score:
            best_score = score
            best_doctor = doc

    return best_doctor


# ─────────────────────────────────────────────
# 5. SIMULATED ANNEALING - SHIFT OPTIMIZER
# ─────────────────────────────────────────────

def shift_optimizer(doctors, num_slots=3, iterations=500):
    """
    Simulated Annealing to optimize doctor shift assignments.
    Minimizes coverage gaps and maximizes specialization coverage.
    """
    slots = ["Morning", "Afternoon", "Night"]

    # Initial random assignment
    assignment = {doc.name: random.choice(slots) for doc in doctors}

    def cost(assign):
        """Lower is better: penalize unbalanced shifts"""
        slot_counts = defaultdict(int)
        for slot in assign.values():
            slot_counts[slot] += 1
        variance = sum((count - len(doctors)/num_slots)**2 for count in slot_counts.values())
        # Penalize if no emergency doc on night shift
        night_docs = [d for d in doctors if assign[d.name] == "Night" and "Emergency" in d.specialization]
        penalty = 100 if not night_docs else 0
        return variance + penalty

    current_cost = cost(assignment)
    T = 100.0  # Temperature

    for i in range(iterations):
        # Pick random doctor and random new slot
        doc = random.choice(doctors)
        new_slot = random.choice(slots)
        old_slot = assignment[doc.name]
        assignment[doc.name] = new_slot

        new_cost = cost(assignment)
        delta = new_cost - current_cost

        # Accept if better, or probabilistically if worse
        if delta < 0 or random.random() < math.exp(-delta / T):
            current_cost = new_cost
        else:
            assignment[doc.name] = old_slot  # Revert

        T *= 0.995  # Cool down

    return assignment


# ─────────────────────────────────────────────
# 6. ER STATISTICS & REPORTING
# ─────────────────────────────────────────────

class ERStats:
    def __init__(self):
        self.treated = []
        self.wait_times = []

    def record(self, patient, wait_minutes):
        self.treated.append(patient)
        self.wait_times.append(wait_minutes)

    def report(self):
        print("\n" + "="*55)
        print("         📊  ER SESSION STATISTICS")
        print("="*55)
        if not self.treated:
            print("  No patients treated yet.")
            return

        total = len(self.treated)
        avg_wait = sum(self.wait_times) / total if total else 0
        max_wait = max(self.wait_times) if self.wait_times else 0

        severity_counts = defaultdict(int)
        for p in self.treated:
            severity_counts[p.severity] += 1

        print(f"  Total patients treated : {total}")
        print(f"  Average wait time      : {avg_wait:.1f} min")
        print(f"  Max wait time          : {max_wait} min")
        print("\n  Severity Breakdown:")
        for sev in sorted(severity_counts):
            name, icon, _ = SEVERITY_LEVELS[sev]
            print(f"    {icon} Level {sev} ({name:12s}): {severity_counts[sev]} patients")
        print("="*55)


# ─────────────────────────────────────────────
# 7. MAIN ER SIMULATION
# ─────────────────────────────────────────────

class EmergencyRoom:
    def __init__(self, num_beds=10):
        self.triage = TriageQueue()
        self.bed_manager = BedManager(num_beds)
        self.stats = ERStats()

        # Initialize doctors
        self.doctors = [
            Doctor("Smith",   "Emergency Medicine", 15),
            Doctor("Patel",   "Emergency Medicine", 8),
            Doctor("Chen",    "Surgery",            12),
            Doctor("Nguyen",  "Cardiology",         20),
            Doctor("Garcia",  "Pediatrics",         10),
        ]

    def admit_patient(self, name, age, symptoms):
        patient = Patient(name, age, symptoms)
        self.triage.add_patient(patient)
        return patient

    def process_next_patient(self):
        if self.triage.is_empty():
            print("\n  ℹ️  No patients in queue.")
            return None

        patient = self.triage.next_patient()
        print(f"\n  🏥 Processing: {patient}")

        # Assign bed
        bed = self.bed_manager.assign_bed(patient)
        if bed:
            print(f"  🛏️  Assigned to Bed {bed}")
        else:
            print("  ⚠️  No beds available! Patient in waiting area.")

        # Assign doctor
        doctor = assign_best_doctor(patient, self.doctors)
        if doctor:
            wait = patient.wait_time_minutes()
            doctor.assign_patient(patient)
            self.stats.record(patient, wait)
            print(f"  👨‍⚕️  Assigned to Dr. {doctor.name} ({doctor.specialization})")
            print(f"  ⏱️  Wait time: {wait} min")
        else:
            print("  ⚠️  All doctors busy. Patient waiting for doctor.")

        return patient

    def show_queue(self):
        patients = self.triage.all_patients()
        print(f"\n  📋 TRIAGE QUEUE ({len(patients)} patients):")
        if not patients:
            print("    [Queue empty]")
        for i, p in enumerate(patients, 1):
            sev_name, icon, max_wait = SEVERITY_LEVELS[p.severity]
            print(f"    {i}. {icon} {p.name:15s} | {sev_name:12s} | Max wait: {max_wait} min")

    def show_doctors(self):
        print("\n  👨‍⚕️  DOCTOR STATUS:")
        for doc in self.doctors:
            print(f"    {doc}")

    def show_shift_plan(self):
        print("\n  📅 OPTIMIZED SHIFT PLAN (Simulated Annealing):")
        assignment = shift_optimizer(self.doctors)
        shifts = defaultdict(list)
        for name, slot in assignment.items():
            shifts[slot].append(name)
        for slot in ["Morning", "Afternoon", "Night"]:
            docs = ", ".join(f"Dr. {n}" for n in shifts[slot])
            print(f"    {slot:12s}: {docs}")

    def run_demo(self):
        """Full demo run"""
        print("\n" + "="*55)
        print("   🏥  HOSPITAL EMERGENCY ROOM OPTIMIZER")
        print("        AI/ML + Search Techniques")
        print("="*55)

        # --- Admit patients ---
        print("\n📥 ADMITTING PATIENTS...\n" + "-"*40)
        patients_data = [
            ("Alice Kumar",  34, ["chest pain", "difficulty breathing"]),
            ("Bob Singh",    72, ["mild fever", "sprain"]),
            ("Carol Mehta",  5,  ["high fever", "ear pain"]),
            ("David Rao",    45, ["severe bleeding"]),
            ("Eva Sharma",   28, ["abdominal pain"]),
            ("Frank Nair",   60, ["common cold"]),
            ("Grace Thomas", 80, ["stroke symptoms"]),
            ("Harish Pillai",22, ["broken bone"]),
        ]

        for name, age, symptoms in patients_data:
            self.admit_patient(name, age, symptoms)
            time.sleep(0.05)

        # --- Show queue ---
        self.show_queue()

        # --- Process all patients ---
        print("\n\n🔄 PROCESSING PATIENTS (Priority Order)...\n" + "-"*40)
        while not self.triage.is_empty():
            self.process_next_patient()
            time.sleep(0.1)

        # --- Bed status ---
        self.bed_manager.display()

        # --- Doctor status ---
        self.show_doctors()

        # --- Shift optimization ---
        self.show_shift_plan()

        # --- Stats report ---
        self.stats.report()


# ─────────────────────────────────────────────
# 8. INTERACTIVE MENU
# ─────────────────────────────────────────────

def interactive_mode(er):
    while True:
        print("\n" + "─"*45)
        print("  🏥  ER OPTIMIZER — MENU")
        print("─"*45)
        print("  1. Admit new patient")
        print("  2. Process next patient")
        print("  3. Show triage queue")
        print("  4. Show doctor status")
        print("  5. Show bed status")
        print("  6. Show shift plan")
        print("  7. Show statistics")
        print("  0. Exit")
        print("─"*45)
        choice = input("  Enter choice: ").strip()

        if choice == "1":
            name = input("  Patient name: ").strip()
            try:
                age = int(input("  Age: "))
            except ValueError:
                print("  Invalid age."); continue
            syms = input("  Symptoms (comma-separated): ").strip().split(",")
            syms = [s.strip() for s in syms if s.strip()]
            er.admit_patient(name, age, syms)

        elif choice == "2":
            er.process_next_patient()

        elif choice == "3":
            er.show_queue()

        elif choice == "4":
            er.show_doctors()

        elif choice == "5":
            er.bed_manager.display()

        elif choice == "6":
            er.show_shift_plan()

        elif choice == "7":
            er.stats.report()

        elif choice == "0":
            print("\n  👋 Exiting ER Optimizer. Goodbye!\n")
            break
        else:
            print("  ❌ Invalid choice.")


# ─────────────────────────────────────────────
# ENTRY POINT
# ─────────────────────────────────────────────

if __name__ == "__main__":
    er = EmergencyRoom(num_beds=10)

    print("\nChoose mode:")
    print("  1. Run full demo (recommended first time)")
    print("  2. Interactive mode")
    mode = input("Enter 1 or 2: ").strip()

    if mode == "1":
        er.run_demo()
    elif mode == "2":
        interactive_mode(er)
    else:
        print("Invalid choice. Running demo by default.")
        er.run_demo()


Choose mode:
  1. Run full demo (recommended first time)
  2. Interactive mode


Enter 1 or 2:  2



─────────────────────────────────────────────
  🏥  ER OPTIMIZER — MENU
─────────────────────────────────────────────
  1. Admit new patient
  2. Process next patient
  3. Show triage queue
  4. Show doctor status
  5. Show bed status
  6. Show shift plan
  7. Show statistics
  0. Exit
─────────────────────────────────────────────


  Enter choice:  1
  Patient name:  swathi
  Age:  34
  Symptoms (comma-separated):  chest pain, swelling



  ✅ Admitted: [P001] swathi (age 34) | 🔴 CRITICAL | Symptoms: chest pain, swelling

─────────────────────────────────────────────
  🏥  ER OPTIMIZER — MENU
─────────────────────────────────────────────
  1. Admit new patient
  2. Process next patient
  3. Show triage queue
  4. Show doctor status
  5. Show bed status
  6. Show shift plan
  7. Show statistics
  0. Exit
─────────────────────────────────────────────


...


  📅 OPTIMIZED SHIFT PLAN (Simulated Annealing):
    Morning     : Dr. Patel, Dr. Chen, Dr. Nguyen
    Afternoon   : Dr. Garcia
    Night       : Dr. Smith

─────────────────────────────────────────────
  🏥  ER OPTIMIZER — MENU
─────────────────────────────────────────────
  1. Admit new patient
  2. Process next patient
  3. Show triage queue
  4. Show doctor status
  5. Show bed status
  6. Show shift plan
  7. Show statistics
  0. Exit
─────────────────────────────────────────────


  Enter choice:  7



         📊  ER SESSION STATISTICS
  Total patients treated : 1
  Average wait time      : 1.0 min
  Max wait time          : 1 min

  Severity Breakdown:
    🔴 Level 1 (CRITICAL    ): 1 patients

─────────────────────────────────────────────
  🏥  ER OPTIMIZER — MENU
─────────────────────────────────────────────
  1. Admit new patient
  2. Process next patient
  3. Show triage queue
  4. Show doctor status
  5. Show bed status
  6. Show shift plan
  7. Show statistics
  0. Exit
─────────────────────────────────────────────


  Enter choice:  0



  👋 Exiting ER Optimizer. Goodbye!

